---
title: "Lecture 12: RNLA Case Study Part 1"
author: "Jamie Haddock"
format:
    revealjs:
        output-file: Lecture12_slides
        slide-number: true
        preview-links: auto
        logo: figs/hmc.png
        css: input/slides.css
        incremental: true
        smaller: true
        code-fold: true
        embed-resources: true
        html-math-method: katex
    html:
        code-fold: true
        embed-resources: true
        output-file: Lecture12
        code-links:
          - text: "Open in Colab"
            href: "https://colab.research.google.com/github/jamiehadd/Math-151-Probability/blob/main/12_rnla_case_study_1.ipynb"
            icon: "laptop"
    pdf:
        documentclass: article
        toc: true
        number-sections: true
        output-file: Lecture12
        geometry:
          - top=1in
          - left=1in
          - bottom=1in
          - right=1in
format-links: false
jupyter: python3
colab:
  gh-user: "jamiehadd"
  gh-repo: "Math-151-Probability"
filters:
  - input/remove-pause.lua
  - colab
execute:
  echo: true
  eval: true
---

## Randomized Kaczmarz: with vs. without replacement

Standard randomized Kaczmarz draws a fresh row index $I_k\sim p$ at *every* iteration, without consideration for past draws -- **sampling with replacement**. A natural alternative cycles through a random reordering of the rows each epoch, never repeating a row until every row has been used -- **sampling without replacement**. Conditional probability lets us see these are different.

. . .

$P_{\text{with replacement}}(I_2 = 7 | I_1 = 7) = p_7$

$P_{\text{without replacement}}(I_2 = 7 | I_1 = 7) = 0$

With replacement, conditioning on $I_1$ doesn't move the probability for $I_2$ at all. Without replacement, it moves it all the way to zero. 

## Randomized Kaczmarz, revisited

Now we can say precisely what was hinted at earlier: sampling rows **with replacement** makes consecutive draws *independent* -- $P(I_2=i\mid I_1=j)=p_i=P(I_2=i)$ for every $j$. Sampling **without replacement** makes them *dependent*: $P(I_2=i\mid I_1=i)=0\neq p_i$.

## Collision Probability in Randomized Kaczmarz

Suppose two consecutive iterations sample rows $I_1,I_2$ **independently**, each $\sim p$ (with replacement, as we now know to call it). What's the probability the algorithm samples the **same row twice in a row** -- a *collision*?

![](figs/RK_collision_plot.jpeg){width=500}

---

::: {.callout-warning icon=false}
## Theorem: Collision probability
$$P(I_1=I_2) = \sum_{i=1}^m p_i^2.$$
:::

<details><summary>Proof:</summary>
Condition on $I_1$, applying the law of total probability with partitioning on the value of $I_1$: $$P(I_1=I_2) = \sum_{i=1}^m P(I_2=i\mid I_1=i)\,P(I_1=i) = \sum_{i=1}^m P(I_2=i)\,p_i = \sum_{i=1}^m p_i\cdot p_i = \sum_{i=1}^m p_i^2,$$ using independence of $I_1,I_2$ for $P(I_2=i\mid I_1=i)=P(I_2=i)=p_i$. $\blacksquare$
</details>

**Why we care:** if row $i$ is sampled twice in a row, the second projection doesn't update the iterate -- the iterate already satisfies $a_i^\top x=b_i$ exactly after the first projection, so the second step makes *zero* progress toward $x^\star$. A high collision probability means wasted iterations.

. . .

::: {.callout-warning icon=false}
## Fact
The collision probability $\sum_i p_i^2$ is **minimized** by the *uniform* distribution $p_i=1/m$ (giving $\sum_i p_i^2=1/m$, the smallest possible value: $1=(\sum_i p_i)^2 \le m\sum_i p_i^2$ by Cauchy-Schwarz), and **maximized** at $1$ when all the probability mass sits on a single row. So concentrating the sampling distribution on a subset of rows comes at the cost of more wasted iterations.
:::

## Is the randomized Kaczmarz error discrete?

Recall randomized Kaczmarz (RK), solving $Ax = b$: at each step we sample a row index
$I \in \{1,\dots,m\}$ at random and update
$$
x_{k+1} = x_k + \frac{b_I - a_I^\top x_k}{\lVert a_I \rVert^2}\, a_I .
$$

$I$ is clearly a **discrete** RV — only $m$ possible values.

. . .

But consider $E = \lVert x_{k+1} - x^\star \rVert$, the error *after* this step. For fixed
$x_k$, $E$ looks continuous: it's a norm of a real vector, and could in principle be any
real number.

. . .

::: {.callout-tip}
## Functions of discrete RVs are discrete!
$E$ is a **function of the discrete RV $I$**: $E = h(I)$ for some function $h$. Since $I$
takes only $m$ values, $E$ *also* has a range of at most $m$ values — even though each of
those $m$ values is an arbitrary real number, not an integer. **Discreteness is
about the size of the range, not about the values themselves.**
:::

## LOTUS example 2: randomized Kaczmarz, revisited

Recall randomized Kaczmarz (RK), solving $Ax = b$: at each step we sample a row index
$I \in \{1,\dots,m\}$ at random and update
$$
x_{k+1} = x_k + \frac{b_I - a_I^\top x_k}{\lVert a_I \rVert^2}\, a_I .
$$

A popular distribution over the rows is the **row-norm
PMF** $p_I(i) = \|a_i\|^2/\|A\|_F^2$. We'll see why now!

---

Fix the current iterate $x_k$ and let $v = x_k - x^\star$.
LOTUS lets us compute the expected value of *any* function of $I$ straight from $p_I$ —
for instance
$$
E\!\left[\frac{(a_I^\top v)^2}{\|a_I\|^2}\right] = \sum_{i=1}^m \frac{(a_i^\top v)^2}{\|a_i\|^2}\, p_I(i).
$$

. . .

Plugging in $p_I(i) = \|a_i\|^2/\|A\|_F^2$, the $\|a_i\|^2$ cancels:
$$
\sum_{i=1}^m \frac{(a_i^\top v)^2}{\|a_i\|^2}\cdot\frac{\|a_i\|^2}{\|A\|_F^2}
= \frac{1}{\|A\|_F^2}\sum_{i=1}^m (a_i^\top v)^2 = \frac{\|Av\|^2}{\|A\|_F^2}.
$$

If $I$ were sampled **uniformly** instead,
the $\|a_i\|^2$ would *not* cancel, and the final form wouldn't be so nice and simple!

## Linearity + LOTUS: average one-step progress of RK

Recall randomized Kaczmarz's update, and note that the Pythagorean theorem (the update is an orthogonal projection) provides,
$$
\|x_{k+1}-x^\star\|^2 = \|v\|^2 - \frac{(a_I^\top v)^2}{\|a_I\|^2}.
$$
where $v = x_k-x^\star$.

. . .

$\|v\|^2$ is a **constant** given $x_k$ — only $I$ is random. By linearity,
$$
E\big[\|x_{k+1}-x^\star\|^2\big]
= \|v\|^2 - E\!\left[\frac{(a_I^\top v)^2}{\|a_I\|^2}\right]
= \|v\|^2 - \frac{\|Av\|^2}{\|A\|_F^2},
$$
reusing the LOTUS computation from a few slides ago!

---

Since $\|Av\| \ge \sigma_{\min}(A)\|v\|$,
$$
E\big[\|x_{k+1}-x^\star\|^2\big] \;\le\; \left(1 - \frac{\sigma_{\min}(A)^2}{\|A\|_F^2}\right)\|x_k-x^\star\|^2.
$$
Every step shrinks the *expected* squared error by a fixed factor — this a very nice (and very well-cited) result from Thomas Strohmer and Roman Vershynin in 2009!

## Watching the bound play out over many steps

The one-step bound says the expected squared error shrinks by a fixed factor *every*
step. Let's actually run RK and watch $\|x_k-x^\star\|^2$ fall, averaged over many
independent random runs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(151)
m, n = 20, 8
A = rng.standard_normal((m, n))
x_star = rng.standard_normal(n)
b = A @ x_star
row_norm_sq = (A ** 2).sum(axis=1)
p_row_norm = row_norm_sq / row_norm_sq.sum()
sigma_min = np.linalg.svd(A, compute_uv=False)[-1]
frob_sq = row_norm_sq.sum()

num_steps = 60
num_runs = 300
errors_sq = np.zeros((num_runs, num_steps + 1))

for r in range(num_runs):
    x = np.zeros(n)
    errors_sq[r, 0] = np.linalg.norm(x - x_star) ** 2
    idx_sequence = rng.choice(m, size=num_steps, p=p_row_norm)
    for t, i in enumerate(idx_sequence, start=1):
        a_i, b_i = A[i], b[i]
        x = x + (b_i - a_i @ x) / row_norm_sq[i] * a_i
        errors_sq[r, t] = np.linalg.norm(x - x_star) ** 2

mean_error_sq = errors_sq.mean(axis=0)
bound_curve = errors_sq[:, 0].mean() * (1 - sigma_min**2 / frob_sq) ** np.arange(num_steps + 1)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(mean_error_sq, label="average of $\\|x_k-x^\\star\\|^2$ over runs")
ax.semilogy(bound_curve, "--", label="one-step bound, applied repeatedly")
ax.set_xlabel("step $k$")
ax.set_ylabel("squared error (log scale)")
ax.legend()
plt.tight_layout()
plt.show()

## Recursing the randomized Kaczmarz bound

Recall our previous one-step bound, valid conditional on the *current* iterate $x_k$:
$$
E\big[\|x_{k+1}-x^\star\|^2 \mid x_k\big] \le (1-c)\,\|x_k-x^\star\|^2,
\qquad c = \frac{\sigma_{\min}(A)^2}{\|A\|_F^2}.
$$

. . .


That's only a statement about *one* step, conditional on where we happen to be. To
turn it into an unconditional guarantee after $k$ steps, apply the law of total
expectation repeatedly:
$$
\begin{aligned}
E\big[\|x_2-x^\star\|^2\big] &= E\Big[E\big[\|x_2-x^\star\|^2 \mid x_1\big]\Big]
\le E\big[(1-c)\|x_1-x^\star\|^2\big] \\&= (1-c)\,E\big[\|x_1-x^\star\|^2\big]
\le (1-c)^2\|x_0-x^\star\|^2.
\end{aligned}
$$

---

Repeating this argument (tower property, one step at a time) gives, for every $k$,
$$
E\big[\|x_k-x^\star\|^2\big] \le (1-c)^k \|x_0-x^\star\|^2,
$$
— exactly the compounded bound we *plotted* before. Now we
know why "applying the one-step bound repeatedly" is legitimate: it's the law of
total expectation, used $k$ times.

In [ ]:
import numpy as np

rng = np.random.default_rng(151)
m, n = 20, 8
A = rng.standard_normal((m, n))
x_star = rng.standard_normal(n)
b = A @ x_star
row_norm_sq = (A ** 2).sum(axis=1)
p_row_norm = row_norm_sq / row_norm_sq.sum()
sigma_min = np.linalg.svd(A, compute_uv=False)[-1]
frob_sq = row_norm_sq.sum()
c = sigma_min**2 / frob_sq

num_steps = 60
num_runs = 2000
checkpoints = [0, 10, 20, 30, 40, 50, 60]
errors_sq = np.zeros((num_runs, num_steps + 1))

for r in range(num_runs):
    x = np.zeros(n)
    errors_sq[r, 0] = np.linalg.norm(x - x_star) ** 2
    idx_sequence = rng.choice(m, size=num_steps, p=p_row_norm)
    for t, i in enumerate(idx_sequence, start=1):
        a_i, b_i = A[i], b[i]
        x = x + (b_i - a_i @ x) / row_norm_sq[i] * a_i
        errors_sq[r, t] = np.linalg.norm(x - x_star) ** 2

mean_error_sq = errors_sq.mean(axis=0)
initial_error_sq = errors_sq[:, 0].mean()

print(f"{'k':>4} {'empirical E[||x_k-x*||^2]':>28} {'bound (1-c)^k ||x_0-x*||^2':>28}")
for k in checkpoints:
    bound = initial_error_sq * (1 - c) ** k
    print(f"{k:>4} {mean_error_sq[k]:>28.5f} {bound:>28.5f}")

**Revisiting our RK example:** Recall randomized Kaczmarz samples a row index $I_k\in\{1,\dots,m\}$ at every
iteration from a fixed PMF $p_I$ (e.g. the row-norm PMF seen previously), **with
replacement** — each iteration's draw doesn't remove that row from future
consideration.

Because the draws are made independently at each iteration, the joint PMF of the indices at two iterations
factors:
$$
p_{I_1,I_2}(i,j) = p_I(i)\,p_I(j).
$$

---

Contrast this with *without-replacement* row-sampling — there,
removing a row after sampling it means the joint PMF of two draws does **not**
factor this way (the second draw's distribution depends on what the first draw was).

## Example: unbiasedness of randomized trace estimation

Recall the randomized trace estimator: for $A\in\mathbb R^{n\times n}$, draw
$z\in\{-1,+1\}^n$ with i.i.d.\ Rademacher entries and set
$\tau = z^\top A z = \sum_{i,j} z_iz_j A_{ij}$. We'll now prove $E[\tau]=\mathrm{tr}(A)$ -- this is called **unbiasedness**.

. . .

By linearity, $E[\tau] = \sum_{i,j}A_{ij}\,E[z_iz_j]$. Split into diagonal and
off-diagonal terms:

- **Diagonal ($i=j$):** $z_i^2=1$ always, so $E[z_i^2]=1$.
- **Off-diagonal ($i\ne j$):** $z_i,z_j$ independent, so by today's fact,
  $E[z_iz_j]=E[z_i]E[z_j] = 0\cdot 0=0$ (Rademacher has mean $0$).

. . .

$$
E[\tau] = \sum_i A_{ii}\cdot 1 + \sum_{i\ne j} A_{ij}\cdot 0 = \sum_i A_{ii}
= \mathrm{tr}(A).
$$

## Example: continuous RV in randomized trace estimation

Recall the randomized trace estimator: for $A\in\mathbb R^{n\times n}$, draw
$z\in\mathbb R^n$ with i.i.d.\ mean-$0$, variance-$1$, pairwise independent entries
and set $\tau = z^\top A z$. The unbiasedness proof only ever used
$$
E[z_i^2] = 1 \qquad\text{and}\qquad E[z_iz_j] = 0 \ (i\ne j),
$$
which followed from mean $0$, variance $1$, and pairwise independence —
**nothing about $z_i$ being discrete**. So a continuous $z_i$ works exactly as well.

. . .

::: {.callout-tip}
## A continuous alternative
Instead of Rademacher entries $z_i=\pm1$, draw $z_i$ i.i.d.\ from a continuous
distribution with mean $0$ and variance $1$ — for concreteness, the (bell-shaped)
standard Gaussian, whose PDF we'll meet formally in a later lecture. The same proof works exactly, just a different distribution!
:::

---

In [ ]:
rng = np.random.default_rng(151)
d = 50
A = rng.standard_normal((d, d)) + 5 * np.eye(d)
true_trace = np.trace(A)

num_samples = 20_000
Z_rad = rng.choice([-1.0, 1.0], size=(num_samples, d))  # discrete: Rademacher
Z_gauss = rng.standard_normal((num_samples, d))          # continuous: standard Gaussian

tau_rad = np.einsum('si,ij,sj->s', Z_rad, A, Z_rad)
tau_gauss = np.einsum('si,ij,sj->s', Z_gauss, A, Z_gauss)

print(f"true trace(A)                        = {true_trace:.4f}")
print(f"mean of tau, Rademacher z (discrete)  = {tau_rad.mean():.4f}")
print(f"mean of tau, Gaussian z (continuous)  = {tau_gauss.mean():.4f}")